
# 06 – Iteration 3: Product Pipeline Demo

This notebook demonstrates the **end-to-end anomaly scoring pipeline** for Dataset 2
(factures amb consums modificats) using the assets produced in previous notebooks:

- Cleaned dataset (Notebook 01)
- Feature engineering logic (Notebook 02 / `src/features.py`)
- Preprocessor parameters saved in `preprocessor_iter3_params.json` (Notebook 04)
- Trained RandomForest model from Notebook 05

Pipeline shown here:

1. Load the cleaned Dataset 2 (iteration 3 version)
2. Take a small random demo sample
3. Build engineered features on top of the raw fields
4. Select the same feature columns used for model training
5. Reconstruct the preprocessor (imputer + scaler) from saved params
6. Use the trained model to obtain anomaly scores
7. Show the top-N most suspicious records


In [1]:

import os
import sys
import json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

# Allow importing project src/
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from features import build_features  # feature engineering used in Notebook 2

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))
CLEANED_DIR = os.path.join(RESULTS_DIR, "cleaned")
PREPARED_DIR = os.path.join(RESULTS_DIR, "prepared")
MODELS_DIR = os.path.join(RESULTS_DIR, "models")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RESULTS_DIR  :", RESULTS_DIR)
print("CLEANED_DIR  :", CLEANED_DIR)
print("PREPARED_DIR :", PREPARED_DIR)
print("MODELS_DIR   :", MODELS_DIR)


PROJECT_ROOT : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D
RESULTS_DIR  : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results
CLEANED_DIR  : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\cleaned
PREPARED_DIR : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared
MODELS_DIR   : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\models


In [2]:

# 1) Load preprocessor parameters saved in Notebook 04
PARAMS_PATH = os.path.join(PREPARED_DIR, "preprocessor_iter3_params.json")

if not os.path.exists(PARAMS_PATH):
    raise FileNotFoundError(f"Preprocessor params JSON not found: {PARAMS_PATH}")

with open(PARAMS_PATH, "r") as f:
    params = json.load(f)

feature_cols = params["feature_cols"]
imputer_strategy = params.get("imputer_strategy", "median")
scaler_mean = np.array(params["scaler_mean"], dtype=float)
scaler_scale = np.array(params["scaler_scale"], dtype=float)

# Rebuild scaler
scaler = StandardScaler()
scaler.mean_ = scaler_mean
scaler.scale_ = scaler_scale
scaler.n_features_in_ = scaler_mean.shape[0]

# Imputer: we might not have stored statistics, so we handle both cases
imputer_statistics = params.get("imputer_statistics", None)
if imputer_statistics is not None:
    imputer = SimpleImputer(strategy=imputer_strategy)
    imputer.statistics_ = np.array(imputer_statistics, dtype=float)
    print("Reconstructed imputer from saved statistics.")
else:
    # We will fit the imputer later on the demo data if needed
    imputer = None
    print("No imputer statistics in params. Will fit imputer on demo data if needed.")

print("\nLoaded preprocessor params from:", PARAMS_PATH)
print(" - #feature_cols:", len(feature_cols))
print(" - imputer_strategy:", imputer_strategy)


No imputer statistics in params. Will fit imputer on demo data if needed.

Loaded preprocessor params from: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\prepared\preprocessor_iter3_params.json
 - #feature_cols: 7
 - imputer_strategy: median


In [3]:

# 2) Load trained model
MODEL_PATH = os.path.join(MODELS_DIR, "model_iter3_rf.joblib")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

model = joblib.load(MODEL_PATH)
print("Loaded model from:", MODEL_PATH)
print("Model type:", type(model))


Loaded model from: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\models\model_iter3_rf.joblib
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:

# 3) Load cleaned Dataset 2 (iteration 3)
CLEANED_PARQUET_PATH = os.path.join(CLEANED_DIR, "dataset2_cleaned_iter3.parquet")

if not os.path.exists(CLEANED_PARQUET_PATH):
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_PARQUET_PATH}")

df_clean_full = pd.read_parquet(CLEANED_PARQUET_PATH)
print("Full cleaned dataset shape:", df_clean_full.shape)

# Take a small random sample for the demo
n_demo = 1000
if df_clean_full.shape[0] > n_demo:
    df_clean_demo = df_clean_full.sample(n=n_demo, random_state=42)
else:
    df_clean_demo = df_clean_full.copy()

print("Demo subset shape:", df_clean_demo.shape)
df_clean_demo.head()


Full cleaned dataset shape: (21195970, 11)
Demo subset shape: (1000, 11)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840
10906207,7YFBITDLZC2X2EBK,163840,2023-02-01,2023-04-05,COMERCIAL,0801901030,H23VA237979H,0.0,2024-08-10 20:01:44,False,True
9044818,EFBQF7ZISVODMVAK,163840,2024-03-28,2024-05-28,DOMÈSTIC,0801908037,J23OA015349X,0.0,2024-05-02 18:57:13,False,True
15023182,CXXA5WXQT6BTFOJ3,32768,2023-08-02,2023-10-03,DOMÈSTIC,0810101009,P16VA115198P,0.0,2024-03-11 21:33:36,True,False
13280597,75W3DOTAAGXZXCX5,32768,2023-12-11,2024-02-07,COMERCIAL,0801909004,P17VA110405S,0.0,2024-12-09 05:36:19,True,False
5763933,SC72IJ2VIEG22ZHP,163840,2023-06-27,2023-08-28,DOMÈSTIC,0801903015,P18VA123792Q,0.0,2024-11-08 03:45:33,False,True


In [5]:

# 4) Build engineered features for the demo subset
df_feat_demo = build_features(df_clean_demo)
print("Feature-augmented demo shape:", df_feat_demo.shape)
df_feat_demo.head()


Feature-augmented demo shape: (1000, 19)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840,y_anom,datetime,cons_lag1,delta1,meter_mean,meter_std,cons_z_meter,period_hours
14787939,5POZCSY7PYKOIX7Z,163840,2024-09-18,2024-11-18,DOMÈSTIC,0810104019,02070511,0.0,2024-02-28 02:13:31,False,True,1,2024-02-28 02:13:31,NaN,NaN,0.0,NaN,NaN,1464.0
21117040,7Z6MFNPLNQNNDSIV,163840,2022-12-21,2023-02-17,DOMÈSTIC,0801910080,02128928,0.0,2024-10-29 04:15:02,False,True,1,2024-10-29 04:15:02,NaN,NaN,0.0,NaN,NaN,1392.0
13175068,3KYE6ZKVPNRWFHK3,163840,2024-08-06,2024-10-06,GENERAL,0801902118,52005144,0.0,2024-12-04 00:49:18,False,True,1,2024-12-04 00:49:18,NaN,NaN,0.0,NaN,NaN,1464.0
20030437,QPQ7Z5YDM5LA4WYY,163840,2022-11-07,2023-01-05,DOMÈSTIC,0810102004,A16FA259956M,1.0,2024-11-26 07:29:11,False,True,1,2024-11-26 07:29:11,NaN,NaN,1.0,NaN,NaN,1416.0
12468174,VMI4XM7TTOLZXJCZ,163840,2023-08-09,2023-10-09,DOMÈSTIC,0801910034,A16FA288885Z,4.0,2024-10-29 21:51:31,False,True,1,2024-10-29 21:51:31,NaN,NaN,4.0,NaN,NaN,1464.0


In [6]:

# 5) Select the same feature columns used during training
missing_feats = [c for c in feature_cols if c not in df_feat_demo.columns]
if missing_feats:
    print("WARNING: The following expected feature columns are missing in the demo feature table:")
    print(missing_feats)

used_features = [c for c in feature_cols if c in df_feat_demo.columns]
print("Using", len(used_features), "features for the demo:")
print(used_features)

X_demo_raw = df_feat_demo[used_features].values

# If we do not have saved imputer statistics, fit imputer on demo data
if imputer is None:
    imputer = SimpleImputer(strategy=imputer_strategy)
    imputer.fit(X_demo_raw)
    print("Fitted a new imputer on demo data.")

X_demo_imp = imputer.transform(X_demo_raw)
X_demo = scaler.transform(X_demo_imp)

print("X_demo shape:", X_demo.shape)


Using 7 features for the demo:
['period_hours', 'meter_std', 'meter_mean', 'cons_z_meter', 'CONSUMO_REAL', 'cons_lag1', 'delta1']
Fitted a new imputer on demo data.
X_demo shape: (1000, 7)


In [7]:

# 6) Get anomaly scores (probability of y_anom = 1)
if hasattr(model, "predict_proba"):
    scores = model.predict_proba(X_demo)[:, 1]
else:
    scores = model.decision_function(X_demo)

df_out = df_feat_demo.copy()
df_out["anom_score"] = scores

key_cols = [
    "POLISSA_SUBM",
    "NUMEROSERIECONTADOR",
    "FECHA_HORA",
    "CODI_ANOMALIA",
    "y_anom",
    "US_AIGUA_SUBM",
    "SECCIO_CENSAL",
]
key_cols = [c for c in key_cols if c in df_out.columns]

display_cols = key_cols + ["anom_score"]

df_out_sorted = df_out.sort_values("anom_score", ascending=False)

print("Top 20 most suspicious records (by model score):")
df_out_sorted[display_cols].head(20)


Top 20 most suspicious records (by model score):


,POLISSA_SUBM,NUMEROSERIECONTADOR,FECHA_HORA,CODI_ANOMALIA,y_anom,US_AIGUA_SUBM,SECCIO_CENSAL,anom_score
19677995,YWIYQSOVTT7OS5GO,P24VA121713T,2024-11-09 00:54:58,32768,1,DOMÈSTIC,0830101001,1.0
14787939,5POZCSY7PYKOIX7Z,02070511,2024-02-28 02:13:31,163840,1,DOMÈSTIC,0810104019,1.0
21117040,7Z6MFNPLNQNNDSIV,02128928,2024-10-29 04:15:02,163840,1,DOMÈSTIC,0801910080,1.0
13175068,3KYE6ZKVPNRWFHK3,52005144,2024-12-04 00:49:18,163840,1,GENERAL,0801902118,1.0
20030437,QPQ7Z5YDM5LA4WYY,A16FA259956M,2024-11-26 07:29:11,163840,1,DOMÈSTIC,0810102004,1.0
12468174,VMI4XM7TTOLZXJCZ,A16FA288885Z,2024-10-29 21:51:31,163840,1,DOMÈSTIC,0801910034,1.0
14213714,FJJXVKYDGAYPFKGN,A17FA021112E,2024-01-27 10:00:00,32768,1,DOMÈSTIC,0801909095,1.0
18534269,FJJXVKYDGAYPFKGN,A17FA021112E,2024-09-12 15:00:00,32768,1,DOMÈSTIC,0801909095,1.0
19345593,JCVYDVFKE36RWRJW,P23VA152496N,2024-10-23 11:46:33,262144,1,DOMÈSTIC,0821103004,1.0
11942817,GE6BSZJGRD3OSZ57,P23VA149885T,2024-10-03 08:53:41,163840,1,GENERAL,0830101006,1.0


In [8]:

print("Demo pipeline summary:")
print("- Input rows (demo):", df_clean_demo.shape[0])
print("- Engineered features used:", len(used_features))
print("- Anomaly score range:",
      float(df_out['anom_score'].min()),
      "to",
      float(df_out['anom_score'].max()))


Demo pipeline summary:
- Input rows (demo): 1000
- Engineered features used: 7
- Anomaly score range: 0.97 to 1.0
